# GOOD scenario configuration

This notebook builds scenario JSON files for CAISO, NYISO, ERCOT, and PJM using one region-aware configuration block.

PJM uses three state RPS policy cases: baseline, 10 percentage points lower, and 10 percentage points higher.


In [1]:
import json
from pathlib import Path
from copy import deepcopy

# =========================================================
# Scenario builder
# =========================================================

def _normalize_rps_cases(rps_targets=None, rps_cases=None):
    """
    Convert either simple RPS targets or full RPS case dictionaries
    into one standard list used by build_scenarios().

    For single-state regions, rps_targets is enough.
    For PJM, rps_cases can carry a state-level RPS dictionary for each case.
    """

    if rps_cases is not None:
        return deepcopy(rps_cases)

    if rps_targets is None:
        rps_targets = [0.0]

    cases = []

    for rps in rps_targets:
        rps_float = float(rps)
        cases.append({
            "rps_case": f"rps{int(round(rps_float * 100))}",
            "rps_label": f"RPS {int(round(rps_float * 100))}%",
            "rps_ratio": rps_float,
            "rps_adjustment_pp": 0.0,
        })

    return cases


def build_scenarios(
    rps_targets=None,
    rps_cases=None,
    battery_capex_cases=None,
    participation_cases=None,
    state_cases=None,
    non_compliance_capacity=1e12,
    non_compliance_cost=5e-5,
    reserve_margin=0.10,
    reserve_margin_sign=-1,
    reserve_non_compliance_capacity=1e6,
    reserve_non_compliance_cost=1,
    start_id=1,
):
    """
    Build scenario dictionary for GOOD model runs.

    This function supports two RPS designs:

    1. single_state_region
       The scenario uses cfg["rps_ratio"] directly.
       This is appropriate for CAISO, NYISO, and ERCOT when you want
       sensitivity cases such as 60%, 70%, and 80%.

    2. multi_state_region
       The scenario can carry cfg["state_rps_policies"].
       This is appropriate for PJM because PJM contains several states.
       Each PJM RPS case can adjust all state RPS policies at once.
    """

    if battery_capex_cases is None:
        battery_capex_cases = [
            (4.16e-5, "$150/kWh"),
            # (6.95e-5, "$250/kWh"),
        ]

    if participation_cases is None:
        participation_cases = [
            {"group": "Base only", "v1g_share": 0.0,  "v2g_share": 0.0},
            {"group": "V1G",       "v1g_share": 0.25, "v2g_share": 0.0},
            {"group": "V2G",       "v1g_share": 0.0,  "v2g_share": 0.25},
            {"group": "V1G",       "v1g_share": 0.50, "v2g_share": 0.0},
            {"group": "V2G",       "v1g_share": 0.0,  "v2g_share": 0.50},
        ]

    if state_cases is None:
        state_cases = [
            {
                "state": "CA",
                "state_name": "california",
                "model_region": "CAISO",
                "jurisdiction": "CA",
                "region_type": "single_state_region",
            },
        ]

    normalized_rps_cases = _normalize_rps_cases(
        rps_targets=rps_targets,
        rps_cases=rps_cases,
    )

    scenarios = {}
    sid = start_id

    for state_case in state_cases:
        for batt_capex_cost, batt_capex_label in battery_capex_cases:
            for case in participation_cases:
                for rps_case in normalized_rps_cases:

                    # Keep all region metadata from the state case.
                    scenario = deepcopy(state_case)

                    scenario.update({
                        "group": case["group"],
                        "rps_ratio": float(rps_case.get("rps_ratio", 0.0)),
                        "rps_case": rps_case.get("rps_case", "rps_custom"),
                        "rps_label": rps_case.get("rps_label", "Custom RPS case"),

                        "rps_display": rps_case.get("rps_display", rps_case.get("rps_label", "Custom RPS case")),
                        "rps_adjustment_label": rps_case.get("rps_adjustment_label", ""),

                        "rps_adjustment_pp": float(rps_case.get("rps_adjustment_pp", 0.0)),
                        "v1g_share": case["v1g_share"],
                        "v2g_share": case["v2g_share"],
                        "batt_capex_cost": batt_capex_cost,
                        "batt_capex_label": batt_capex_label,

                        # RPS policy settings
                        "rps_non_compliance_capacity": non_compliance_capacity,
                        "rps_non_compliance_cost": non_compliance_cost,

                        # Reserve margin policy settings
                        "reserve_margin": reserve_margin,
                        "reserve_margin_sign": reserve_margin_sign,
                        "reserve_non_compliance_capacity": reserve_non_compliance_capacity,
                        "reserve_non_compliance_cost": reserve_non_compliance_cost,
                    })

                    # PJM uses this field so each scenario can carry its own
                    # state-level RPS dictionary.
                    if "state_rps_policies" in rps_case:
                        scenario["state_rps_policies"] = deepcopy(
                            rps_case["state_rps_policies"]
                        )

                    scenarios[sid] = scenario
                    sid += 1

    return scenarios


In [2]:
# =========================================================
# Region policy configuration
# =========================================================

# Multi-state regions use state-level RPS policies.
#
# Each multi-state region has three cases:
#   1. State RPS baseline minus 10 percentage points
#   2. State RPS baseline
#   3. State RPS baseline plus 10 percentage points
#
# States with a zero baseline remain zero in all three cases.


# =========================================================
# State-level RPS adjustment functions
# =========================================================

def make_state_rps_policy_case(
    base_policies,
    adjustment_pp=0.0,
    lower_bound=0.0,
    upper_bound=1.0,
    zero_states_stay_zero=True,
    eps=1e-9,
):
    """
    Create one state-level RPS policy case.

    adjustment_pp is an absolute percentage-point adjustment.

    Examples:
        0.40 + 0.10 = 0.50
        0.08 - 0.10 = 0.00 after clipping

    States with a zero baseline remain zero by default.
    """

    adjusted = {}

    for state, policy in base_policies.items():
        new_policy = deepcopy(policy)

        base_ratio = float(
            new_policy.get("rps_ratio", 0.0)
        )

        if abs(base_ratio) < eps:
            base_ratio = 0.0

        if zero_states_stay_zero and base_ratio <= 0.0:
            new_ratio = 0.0

        else:
            new_ratio = (
                base_ratio
                + float(adjustment_pp)
            )

            new_ratio = max(
                lower_bound,
                new_ratio,
            )

            new_ratio = min(
                upper_bound,
                new_ratio,
            )

            if abs(new_ratio) < eps:
                new_ratio = 0.0

        new_policy["rps_ratio"] = round(
            new_ratio,
            6,
        )

        adjusted[state] = new_policy

    return adjusted


def make_state_rps_cases(base_policies):
    """
    Create the low, baseline, and high RPS cases
    for one multi-state model region.
    """

    return [
        {
            "rps_case": "rps_minus10",
            "rps_label": (
                "State RPS minus 10 percentage points"
            ),
            "rps_display": "RPS −10 pp",
            "rps_adjustment_label": "−10 pp",

            # Metadata describing the policy adjustment
            "rps_ratio": -0.10,
            "rps_adjustment_pp": -0.10,

            "state_rps_policies": (
                make_state_rps_policy_case(
                    base_policies,
                    adjustment_pp=-0.10,
                )
            ),
        },

        {
            "rps_case": "rps_base",
            "rps_label": "State RPS baseline",
            "rps_display": "RPS baseline",
            "rps_adjustment_label": "baseline",

            # Metadata describing the policy adjustment
            "rps_ratio": 0.00,
            "rps_adjustment_pp": 0.00,

            "state_rps_policies": (
                make_state_rps_policy_case(
                    base_policies,
                    adjustment_pp=0.00,
                )
            ),
        },

        {
            "rps_case": "rps_plus10",
            "rps_label": (
                "State RPS plus 10 percentage points"
            ),
            "rps_display": "RPS +10 pp",
            "rps_adjustment_label": "+10 pp",

            # Metadata describing the policy adjustment
            "rps_ratio": 0.10,
            "rps_adjustment_pp": 0.10,

            "state_rps_policies": (
                make_state_rps_policy_case(
                    base_policies,
                    adjustment_pp=0.10,
                )
            ),
        },
    ]


# =========================================================
# PJM baseline state RPS policies
# =========================================================

# DC is intentionally set to zero because DC's RPS is
# primarily a retail REC compliance policy.
#
# The physical GOOD PJM graph does not have sufficient
# DC generation assets to apply it as an in-region
# physical generation constraint.

PJM_STATE_RPS_POLICIES_BASE = {
    "DE": {"rps_ratio": 0.28},
    "IL": {"rps_ratio": 0.40},
    "IN": {"rps_ratio": 0.00},
    "KY": {"rps_ratio": 0.00},
    "MD": {"rps_ratio": 0.50},
    "MI": {"rps_ratio": 0.20},
    "NJ": {"rps_ratio": 0.40},
    "NC": {"rps_ratio": 0.125},
    "OH": {"rps_ratio": 0.085},
    "PA": {"rps_ratio": 0.08},
    "TN": {"rps_ratio": 0.00},
    "VA": {"rps_ratio": 0.31},
    "WV": {"rps_ratio": 0.00},
    "DC": {"rps_ratio": 0.00},
}


# Backward-compatible name for other notebooks.

PJM_STATE_RPS_POLICIES = deepcopy(
    PJM_STATE_RPS_POLICIES_BASE
)


# =========================================================
# MAPP baseline state RPS policies
# =========================================================

MAPP_STATE_RPS_POLICIES_BASE = {
    "MT": {"rps_ratio": 0.00},
    "ND": {"rps_ratio": 0.00},
    "SD": {"rps_ratio": 0.00},
}


# =========================================================
# MISO baseline state RPS policies
# =========================================================

MISO_STATE_RPS_POLICIES_BASE = {
    "AR": {"rps_ratio": 0.00},
    "IA": {"rps_ratio": 0.005822},
    "IL": {"rps_ratio": 0.40},
    "IN": {"rps_ratio": 0.00},
    "KY": {"rps_ratio": 0.00},
    "LA": {"rps_ratio": 0.00},
    "MI": {"rps_ratio": 0.20},
    "MN": {"rps_ratio": 0.25},
    "MO": {"rps_ratio": 0.15},
    "MS": {"rps_ratio": 0.00},
    "TX": {"rps_ratio": 0.00},
    "WI": {"rps_ratio": 0.10},
}


# =========================================================
# ISO New England baseline state RPS policies
# =========================================================

ISO_NE_STATE_RPS_POLICIES_BASE = {
    "CT": {"rps_ratio": 0.33},
    "MA": {"rps_ratio": 0.47},
    "ME": {"rps_ratio": 0.80},
    "NH": {"rps_ratio": 0.23},
    "RI": {"rps_ratio": 0.72},
    "VT": {"rps_ratio": 0.80},
}


# =========================================================
# SERC East baseline state RPS policies
# =========================================================

SERC_E_STATE_RPS_POLICIES_BASE = {
    "NC": {"rps_ratio": 0.125},
    "SC": {"rps_ratio": 0.00},
    "VA": {"rps_ratio": 0.31},
}


# =========================================================
# SERC North baseline state RPS policies
# =========================================================

SERC_N_STATE_RPS_POLICIES_BASE = {
    "AL": {"rps_ratio": 0.00},
    "GA": {"rps_ratio": 0.00},
    "KY": {"rps_ratio": 0.00},
    "MO": {"rps_ratio": 0.15},
    "MS": {"rps_ratio": 0.00},
    "NC": {"rps_ratio": 0.125},
    "TN": {"rps_ratio": 0.00},
    "VA": {"rps_ratio": 0.31},
}


# =========================================================
# SERC Southeast baseline state RPS policies
# =========================================================

SERC_SE_STATE_RPS_POLICIES_BASE = {
    "AL": {"rps_ratio": 0.00},
    "GA": {"rps_ratio": 0.00},
    "MS": {"rps_ratio": 0.00},
}


# =========================================================
# SPP baseline state RPS policies
# =========================================================

SPP_STATE_RPS_POLICIES_BASE = {
    "AR": {"rps_ratio": 0.00},
    "KS": {"rps_ratio": 0.00},
    "LA": {"rps_ratio": 0.00},
    "MO": {"rps_ratio": 0.15},
    "MT": {"rps_ratio": 0.00},
    "ND": {"rps_ratio": 0.00},
    "NE": {"rps_ratio": 0.00},
    "NM": {"rps_ratio": 0.50},
    "OK": {"rps_ratio": 0.00},
    "SD": {"rps_ratio": 0.00},
    "TX": {"rps_ratio": 0.00},
    "WY": {"rps_ratio": 0.00},
}


# =========================================================
# NWPP baseline state RPS policies
# =========================================================

NWPP_STATE_RPS_POLICIES_BASE = {
    "CA": {"rps_ratio": 0.60},
    "ID": {"rps_ratio": 0.00},
    "MT": {"rps_ratio": 0.00},
    "NV": {"rps_ratio": 0.50},
    "OR": {"rps_ratio": 0.221780},
    "UT": {"rps_ratio": 0.00},
    "WA": {"rps_ratio": 0.15},
}


# =========================================================
# RMRG baseline state RPS policies
# =========================================================

RMRG_STATE_RPS_POLICIES_BASE = {
    "CO": {"rps_ratio": 0.238804},
    "WY": {"rps_ratio": 0.00},
}


# =========================================================
# SRSG baseline state RPS policies
# =========================================================

SRSG_STATE_RPS_POLICIES_BASE = {
    "AZ": {"rps_ratio": 0.15},
    "CA": {"rps_ratio": 0.60},
    "NM": {"rps_ratio": 0.50},
}


# =========================================================
# Generate RPS cases for every multi-state region
# =========================================================

PJM_STATE_RPS_CASES = make_state_rps_cases(
    PJM_STATE_RPS_POLICIES_BASE
)

MAPP_STATE_RPS_CASES = make_state_rps_cases(
    MAPP_STATE_RPS_POLICIES_BASE
)

MISO_STATE_RPS_CASES = make_state_rps_cases(
    MISO_STATE_RPS_POLICIES_BASE
)

ISO_NE_STATE_RPS_CASES = make_state_rps_cases(
    ISO_NE_STATE_RPS_POLICIES_BASE
)

SERC_E_STATE_RPS_CASES = make_state_rps_cases(
    SERC_E_STATE_RPS_POLICIES_BASE
)

SERC_N_STATE_RPS_CASES = make_state_rps_cases(
    SERC_N_STATE_RPS_POLICIES_BASE
)

SERC_SE_STATE_RPS_CASES = make_state_rps_cases(
    SERC_SE_STATE_RPS_POLICIES_BASE
)

SPP_STATE_RPS_CASES = make_state_rps_cases(
    SPP_STATE_RPS_POLICIES_BASE
)

NWPP_STATE_RPS_CASES = make_state_rps_cases(
    NWPP_STATE_RPS_POLICIES_BASE
)

RMRG_STATE_RPS_CASES = make_state_rps_cases(
    RMRG_STATE_RPS_POLICIES_BASE
)

SRSG_STATE_RPS_CASES = make_state_rps_cases(
    SRSG_STATE_RPS_POLICIES_BASE
)


# =========================================================
# State retirement policies
# =========================================================

PJM_STATE_RETIREMENT_POLICIES = {
    "IL": {
        "coal": 0.0,
        "oil": 0.0,
    },
    "VA": {
        "coal": 0.0,
        "oil": 0.0,
    },
}


# =========================================================
# Helper for multi-state region run configurations
# =========================================================

def make_multi_state_region_run_config(
    model_region,
    base_policies,
    rps_cases,
    included_nodes,
    output_json,
    state_retirement_policies=None,
):
    """
    Create one REGION_RUN_CONFIGS entry for a
    multi-state model region.
    """

    return {
        "state_case": {
            "state": model_region,
            "state_name": (
                model_region
                .lower()
                .replace("-", "_")
            ),
            "model_region": model_region,
            "jurisdiction": model_region,
            "region_type": "multi_state_region",
            "included_states": list(
                base_policies.keys()
            ),
            "included_nodes": list(
                included_nodes
            ),
        },

        "rps_mode": "state_policy_scenarios",

        # Each scenario carries its own state-level
        # RPS policy dictionary.
        "rps_cases": deepcopy(rps_cases),

        "output_json": output_json,

        # Do not pass one fixed policy dictionary.
        "state_rps_policies": None,

        "state_retirement_policies": (
            deepcopy(state_retirement_policies)
            if state_retirement_policies is not None
            else None
        ),
    }


# =========================================================
# Region run configurations
# =========================================================

REGION_RUN_CONFIGS = {
    "CAISO": {
        "state_case": {
            "state": "CA",
            "state_name": "california",
            "model_region": "CAISO",
            "jurisdiction": "CA",
            "region_type": "single_state_region",
        },
        "rps_mode": "scenario_targets",
        "rps_targets": [
            0.50,
            0.60,
            0.70,
        ],
        "output_json": (
            "Examples/scenarios_2030_CAISO.json"
        ),
        "state_rps_policies": None,
        "state_retirement_policies": None,
    },

    "NYISO": {
        "state_case": {
            "state": "NY",
            "state_name": "newyork",
            "model_region": "NYISO",
            "jurisdiction": "NY",
            "region_type": "single_state_region",
        },
        "rps_mode": "scenario_targets",
        "rps_targets": [
            0.60,
            0.70,
            0.80,
        ],
        "output_json": (
            "Examples/scenarios_2030_NYISO.json"
        ),
        "state_rps_policies": None,
        "state_retirement_policies": None,
    },

    "ERCOT": {
        "state_case": {
            "state": "TX",
            "state_name": "texas",
            "model_region": "ERCOT",
            "jurisdiction": "TX",
            "region_type": "single_state_region",
        },
        "rps_mode": "scenario_targets",
        "rps_targets": [
            0.00,
            0.50,
            0.60,
        ],
        "output_json": (
            "Examples/scenarios_2030_ERCOT.json"
        ),
        "state_rps_policies": None,
        "state_retirement_policies": None,
    },

    "PJM": make_multi_state_region_run_config(
        model_region="PJM",
        base_policies=PJM_STATE_RPS_POLICIES_BASE,
        rps_cases=PJM_STATE_RPS_CASES,
        included_nodes=[
            "PJM_AP",
            "PJM_ATSI",
            "PJM_COMD",
            "PJM_Dom",
            "PJM_EMAC",
            "PJM_PENE",
            "PJM_SMAC",
            "PJM_WMAC",
            "PJM_West",
        ],
        output_json=(
            "Examples/scenarios_2030_PJM.json"
        ),
        state_retirement_policies=(
            PJM_STATE_RETIREMENT_POLICIES
        ),
    ),

    "FRCC": {
        "state_case": {
            "state": "FL",
            "state_name": "florida",
            "model_region": "FRCC",
            "jurisdiction": "FL",
            "region_type": "single_state_region",
        },
        "rps_mode": "scenario_targets",
        "rps_targets": [
            0.00,
            0.50,
            0.60,
        ],
        "output_json": (
            "Examples/scenarios_2030_FRCC.json"
        ),
        "state_rps_policies": None,
        "state_retirement_policies": None,
    },

    "MAPP": make_multi_state_region_run_config(
        model_region="MAPP",
        base_policies=MAPP_STATE_RPS_POLICIES_BASE,
        rps_cases=MAPP_STATE_RPS_CASES,
        included_nodes=[
            "MIS_MAPP",
        ],
        output_json=(
            "Examples/scenarios_2030_MAPP.json"
        ),
    ),

    "MISO": make_multi_state_region_run_config(
        model_region="MISO",
        base_policies=MISO_STATE_RPS_POLICIES_BASE,
        rps_cases=MISO_STATE_RPS_CASES,
        included_nodes=[
            "MIS_IL",
            "MIS_INKY",
            "MIS_IA",
            "MIS_MIDA",
            "MIS_LMI",
            "MIS_MO",
            "MIS_WUMS",
            "MIS_MNWI",
            "MIS_WOTA",
            "MIS_AMSO",
            "MIS_AR",
            "MIS_D_MS",
            "MIS_LA",
        ],
        output_json=(
            "Examples/scenarios_2030_MISO.json"
        ),
    ),

    "ISO-NE": make_multi_state_region_run_config(
        model_region="ISO-NE",
        base_policies=ISO_NE_STATE_RPS_POLICIES_BASE,
        rps_cases=ISO_NE_STATE_RPS_CASES,
        included_nodes=[
            "NENG_CT",
            "NENGREST",
            "NENG_ME",
        ],
        output_json=(
            "Examples/scenarios_2030_ISO-NE.json"
        ),
    ),

    "SERC-E": make_multi_state_region_run_config(
        model_region="SERC-E",
        base_policies=SERC_E_STATE_RPS_POLICIES_BASE,
        rps_cases=SERC_E_STATE_RPS_CASES,
        included_nodes=[
            "S_VACA",
        ],
        output_json=(
            "Examples/scenarios_2030_SERC-E.json"
        ),
    ),

    "SERC-N": make_multi_state_region_run_config(
        model_region="SERC-N",
        base_policies=SERC_N_STATE_RPS_POLICIES_BASE,
        rps_cases=SERC_N_STATE_RPS_CASES,
        included_nodes=[
            "S_C_KY",
            "S_D_AECI",
            "S_C_TVA",
        ],
        output_json=(
            "Examples/scenarios_2030_SERC-N.json"
        ),
    ),

    "SERC-SE": make_multi_state_region_run_config(
        model_region="SERC-SE",
        base_policies=SERC_SE_STATE_RPS_POLICIES_BASE,
        rps_cases=SERC_SE_STATE_RPS_CASES,
        included_nodes=[
            "S_SOU",
        ],
        output_json=(
            "Examples/scenarios_2030_SERC-SE.json"
        ),
    ),

    "SPP": make_multi_state_region_run_config(
        model_region="SPP",
        base_policies=SPP_STATE_RPS_POLICIES_BASE,
        rps_cases=SPP_STATE_RPS_CASES,
        included_nodes=[
            "SPP_NEBR",
            "SPP_N",
            "SPP_WEST",
            "SPP_SPS",
            "SPP_WAUE",
        ],
        output_json=(
            "Examples/scenarios_2030_SPP.json"
        ),
    ),

    "NWPP": make_multi_state_region_run_config(
        model_region="NWPP",
        base_policies=NWPP_STATE_RPS_POLICIES_BASE,
        rps_cases=NWPP_STATE_RPS_CASES,
        included_nodes=[
            "WECC_MT",
            "WEC_BANC",
            "WECC_ID",
            "WECC_NNV",
            "WECC_SNV",
            "WECC_UT",
            "WECC_PNW",
        ],
        output_json=(
            "Examples/scenarios_2030_NWPP.json"
        ),
    ),

    "RMRG": make_multi_state_region_run_config(
        model_region="RMRG",
        base_policies=RMRG_STATE_RPS_POLICIES_BASE,
        rps_cases=RMRG_STATE_RPS_CASES,
        included_nodes=[
            "WECC_CO",
            "WECC_WY",
        ],
        output_json=(
            "Examples/scenarios_2030_RMRG.json"
        ),
    ),

    "SRSG": make_multi_state_region_run_config(
        model_region="SRSG",
        base_policies=SRSG_STATE_RPS_POLICIES_BASE,
        rps_cases=SRSG_STATE_RPS_CASES,
        included_nodes=[
            "WECC_AZ",
            "WECC_NM",
            "WECC_IID",
        ],
        output_json=(
            "Examples/scenarios_2030_SRSG.json"
        ),
    ),
}

In [3]:
# =========================================================
# User inputs
# =========================================================

# Change only this line when you switch markets.
# Options: "CAISO", "NYISO", "ERCOT", "PJM"
MODEL_REGIONS = [
    "CAISO",
    "NYISO",
    "ERCOT",
    "PJM",
    "FRCC",
    "MAPP",
    "MISO",
    "ISO-NE",
    "SERC-E",
    "SERC-N",
    "SERC-SE",
    "SPP",
    "NWPP",
    "RMRG",
    "SRSG",
]

BATTERY_CAPEX_CASES = [
    # (4.16e-5, "$150/kWh"),
    (8.6e-5, "$315/kWh"),
]

PARTICIPATION_CASES = [
    {"group": "Base only", "v1g_share": 0.0,  "v2g_share": 0.0},
    {"group": "V1G",       "v1g_share": 0.25, "v2g_share": 0.0},
    {"group": "V2G",       "v1g_share": 0.0,  "v2g_share": 0.25},
    {"group": "V1G",       "v1g_share": 0.50, "v2g_share": 0.0},
    {"group": "V2G",       "v1g_share": 0.0,  "v2g_share": 0.50},
]

# ---------------------------------------------------------
# RPS non-compliance settings
# ---------------------------------------------------------

# Use zero for final hard RPS constraints.
# Use a large value only for debugging infeasibility.
NON_COMPLIANCE_CAPACITY = 15000.0
NON_COMPLIANCE_COST = 5e-5

# ---------------------------------------------------------
# Reserve margin settings
# ---------------------------------------------------------

RESERVE_MARGIN = 0.15
RESERVE_MARGIN_SIGN = -1
RESERVE_NON_COMPLIANCE_CAPACITY = 0.0
RESERVE_NON_COMPLIANCE_COST = 1e6


In [4]:
# =========================================================
# Build scenario files for all selected model regions
# =========================================================

unknown_regions = [r for r in MODEL_REGIONS if r not in REGION_RUN_CONFIGS]

if unknown_regions:
    raise ValueError(
        f"Unknown MODEL_REGIONS={unknown_regions}. "
        f"Available model regions are: {list(REGION_RUN_CONFIGS.keys())}"
    )

SCENARIOS_BY_MODEL_REGION = {}
SCENARIO_OUTPUTS = {}

for model_region in MODEL_REGIONS:
    region_cfg = REGION_RUN_CONFIGS[model_region]

    scenarios = build_scenarios(
        rps_targets=region_cfg.get("rps_targets"),
        rps_cases=region_cfg.get("rps_cases"),
        battery_capex_cases=BATTERY_CAPEX_CASES,
        participation_cases=PARTICIPATION_CASES,
        state_cases=[region_cfg["state_case"]],

        # RPS
        non_compliance_capacity=NON_COMPLIANCE_CAPACITY,
        non_compliance_cost=NON_COMPLIANCE_COST,

        # Reserve margin
        reserve_margin=RESERVE_MARGIN,
        reserve_margin_sign=RESERVE_MARGIN_SIGN,
        reserve_non_compliance_capacity=RESERVE_NON_COMPLIANCE_CAPACITY,
        reserve_non_compliance_cost=RESERVE_NON_COMPLIANCE_COST,
    )

    output_path = Path(region_cfg["output_json"])
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w") as f:
        json.dump(scenarios, f, indent=4)

    SCENARIOS_BY_MODEL_REGION[model_region] = scenarios
    SCENARIO_OUTPUTS[model_region] = str(output_path)

    rps_case_labels = sorted({
        cfg.get("rps_case", "unknown")
        for cfg in scenarios.values()
    })

    print("=" * 80)
    print("Scenario file created")
    print("=" * 80)
    print("Model region:", model_region)
    print("RPS mode:", region_cfg["rps_mode"])
    print("RPS cases:", rps_case_labels)
    print("Number of scenarios:", len(scenarios))
    print("Output:", output_path)

    first_id = min(scenarios)
    print("\nExample scenario:")
    print(json.dumps(scenarios[first_id], indent=4))

    if model_region == "PJM":
        print("\nPJM state RPS cases used:")
        for case in region_cfg["rps_cases"]:
            active = {
                state: policy["rps_ratio"]
                for state, policy in case["state_rps_policies"].items()
                if policy["rps_ratio"] > 0
            }
            print(f"  {case['rps_case']}: {active}")

    print("\n")

print("=" * 80)
print("All requested scenario files were created")
print("=" * 80)
print(json.dumps(SCENARIO_OUTPUTS, indent=4))

Scenario file created
Model region: CAISO
RPS mode: scenario_targets
RPS cases: ['rps50', 'rps60', 'rps70']
Number of scenarios: 15
Output: Examples/scenarios_2030_CAISO.json

Example scenario:
{
    "state": "CA",
    "state_name": "california",
    "model_region": "CAISO",
    "jurisdiction": "CA",
    "region_type": "single_state_region",
    "group": "Base only",
    "rps_ratio": 0.5,
    "rps_case": "rps50",
    "rps_label": "RPS 50%",
    "rps_display": "RPS 50%",
    "rps_adjustment_label": "",
    "rps_adjustment_pp": 0.0,
    "v1g_share": 0.0,
    "v2g_share": 0.0,
    "batt_capex_cost": 8.6e-05,
    "batt_capex_label": "$315/kWh",
    "rps_non_compliance_capacity": 15000.0,
    "rps_non_compliance_cost": 5e-05,
    "reserve_margin": 0.15,
    "reserve_margin_sign": -1,
    "reserve_non_compliance_capacity": 0.0,
    "reserve_non_compliance_cost": 1000000.0
}


Scenario file created
Model region: NYISO
RPS mode: scenario_targets
RPS cases: ['rps60', 'rps70', 'rps80']
Number of